# Teaching Monster — Colab Pro+ GPU Notebook

**Runtime requirement**: GPU (A100 recommended on Pro+), High-RAM  
**Estimated total time**: ~30–60 min  

> Before running: set **Runtime → Change runtime type → GPU (A100)** and optionally enable **High-RAM**.

## 0 · GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime to GPU first!"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU  : {gpu.name}")
print(f"VRAM : {gpu.total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__}")

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone repository from GitHub

In [ ]:
import os

REPO_URL  = "https://github.com/ragiokay/TeachingMonster-released-main.git"
REPO_DIR  = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

## 3 · Install system packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libreoffice poppler-utils
print("System packages ready.")

## 4 · Install Python dependencies

Installs GPU-compatible packages without touching Colab's pre-built torch.

In [ ]:
!pip -q install -U pip
!pip -q install -r requirements-colab-gpu.txt
print("Python packages ready.")

## 5 · Configure Gemini API key

In [ ]:
import getpass
from pathlib import Path

key = getpass.getpass("Paste your GEMINI_API_KEY and press Enter: ")
Path("config/.env").write_text(f"GEMINI_API_KEY={key}\n", encoding="utf-8")
print("config/.env written.")

## 6 · Verify Gemini connectivity

In [ ]:
import yaml
from src.config_schema import AppConfig
from src.gemini_client import GeminiClient

cfg = AppConfig(**yaml.safe_load(open("config/default.yaml", encoding="utf-8")))
client = GeminiClient(cfg.llm)
client.load()
reply = client.generate("Reply with OK only.")
print("Gemini says:", reply[:80])

## 7 · Set topic and persona

Edit the two strings below to change the teaching video content.

In [ ]:
REQUIREMENT_PROMPT = "Self-Attention Mechanism in Transformers"
PERSONA_PROMPT     = "High school student with no calculus background"
OUTPUT_NAME        = "teaching_monster_output.mp4"

print(f"Topic   : {REQUIREMENT_PROMPT}")
print(f"Audience: {PERSONA_PROMPT}")
print(f"Output  : ./output/{OUTPUT_NAME}")

## 8 · Run full pipeline (GPU — no fast-mode flags)

This cell downloads model weights on the first run (~20 min), then generates the video.  
Models loaded:
- **TTS** : `Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice` + Whisper medium  
- **Cursor grouping** : `Qwen/Qwen3-VL-8B-Instruct` (4-bit NF4)  
- **Cursor grounding**: `ByteDance-Seed/UI-TARS-1.5-7B` (4-bit NF4)  
- **LLM** : Gemini 2.5 Flash (cloud)

In [ ]:
import os
# Ensure full-quality mode (no fast-mode shortcuts)
os.environ.pop("TTS_FAST_MODE", None)
os.environ.pop("CURSOR_FAST_MODE", None)

!python -m scripts.T2V_pipeline \
  -r "{REQUIREMENT_PROMPT}" \
  -p "{PERSONA_PROMPT}" \
  -o "{OUTPUT_NAME}" \
  -d ./output

## 9 · Verify output

In [ ]:
import os

video_path = f"./output/{OUTPUT_NAME}"
assert os.path.exists(video_path), f"Video not found at {video_path}"
size_mb = os.path.getsize(video_path) / 1e6
print(f"Video ready: {video_path}  ({size_mb:.1f} MB)")

## 10 · Save outputs to Google Drive

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/TeachingMonster/output"

!mkdir -p "{DRIVE_OUT}"
!cp -f ./output/{OUTPUT_NAME} "{DRIVE_OUT}/"
!ls -lh "{DRIVE_OUT}"
print(f"Saved to Google Drive: {DRIVE_OUT}/{OUTPUT_NAME}")

## 11 · (Optional) Save intermediate artifacts for debugging

In [ ]:
DRIVE_LOG = "/content/drive/MyDrive/TeachingMonster/logs"

!mkdir -p "{DRIVE_LOG}"
!cp -r ./tmp "{DRIVE_LOG}/" 2>/dev/null || true
!cp -r ./tmp_pipeline "{DRIVE_LOG}/" 2>/dev/null || true
!ls -lh "{DRIVE_LOG}"
print("Artifacts saved.")

---

### Troubleshooting

| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Enable **High-RAM** in runtime settings and rerun from Cell 8 |
| `ModuleNotFoundError: qwen_tts` | Restart runtime, then re-run Cell 4 |
| Gemini 429 rate-limit | Set `GEMINI_FALLBACK_MODELS=gemini-2.5-flash,gemini-1.5-flash` before Cell 8 |
| Pipeline hangs at TTS | Check GPU memory with `!nvidia-smi`; consider using staged mode (see below) |

### Staged mode (checkpoint/resume)

If the session disconnects mid-run, use staged mode to resume from where you left off:

```bash
# Run up through outline + slides only
!python -m scripts.T2V_staged -r "..." -p "..." --end-stage 2

# Resume from TTS onward
!python -m scripts.T2V_staged -r "..." -p "..." --start-stage 3
```